# CellPose-SAM (cpsam_v2) on GPU — cell detection for the CNMF hybrid pipeline

**What this does:** runs the heavy Cellpose-SAM model on a projection of your recording, on a free
Colab GPU, to find the cells → outputs a **label mask** you bring back to the local notebook.

**Before you start:** `Runtime -> Change runtime type -> Hardware accelerator -> GPU` (T4 is fine).

**Flow:** install → confirm GPU → upload `_proj_corr.tif` (made locally by
`cnmf_toolkit/cellpose_export_projection.py`) → run at **diameter=0 (native)**; optionally the `6c`
min_size/cellprob sweep → **download the label mask** → use it in the local `_dl-seeding` notebook.

Full end-to-end steps: `cnmf_toolkit/DL_SEEDING_WORKFLOW.md`. On a GPU each run is seconds vs 20+ min on CPU.

In [ ]:
# 1) Install CellPose (v4 = Cellpose-SAM). ~1 min.
!pip -q install cellpose
print('\nInstall done. >>> NOW DO: Runtime -> Restart session <<<  then re-run Cell 1 & 2 and continue.')
print("(pip upgrades numpy; a restart is REQUIRED so the new numpy loads cleanly.)")
print("(You may see a 'numba requires numpy<2.1' pip WARNING — it is harmless; CellPose still runs.")
print(" Do NOT pin numpy<2.1: that pulls a numpy that lacks '_no_nep50_warning' and breaks the import.)")

In [ ]:
# 2) Confirm the GPU is active (this is the whole point).
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('NO GPU -> Runtime > Change runtime type > GPU, then re-run.')

## 3) Upload the projection
Run the cell, then pick `_proj_corr.tif` (and optionally `_proj_mean.tif` / `_proj_max.tif`) from your machine — the files `cnmf_toolkit/cellpose_export_projection.py` wrote to `data/results/comparisons/`.

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose _proj_corr.tif (~1 MB)
print('uploaded:', list(uploaded.keys()))

In [ ]:
# 4) Load the model once + define a run helper.
import numpy as np, tifffile, matplotlib.pyplot as plt
from cellpose import models
from skimage.segmentation import find_boundaries

# cpsam_v2 = Cellpose-SAM with the low-contrast fix (good for low-SNR electrocytes).
# First eval downloads the ~1.2 GB weights (fast on Colab).
model = models.CellposeModel(gpu=True, pretrained_model='cpsam_v2')

def run(proj_file, diameter, min_size=15, cellprob_threshold=0.0):
    """Segment a projection. diameter=0 -> native scale (no resize).
    min_size ↑ drops tiny fragments (fewer splits + tiny junk).
    cellprob_threshold ↑ makes CellPose stricter (fewer marginal junk detections)."""
    img = tifffile.imread(proj_file).astype('float32')
    img = (img - img.min()) / (np.ptp(img) + 1e-9)
    eval_diam = None if (diameter is None or diameter <= 0) else diameter
    masks, flows, styles = model.eval(img, diameter=eval_diam,
                                      min_size=min_size, cellprob_threshold=cellprob_threshold)  # v4: 3 returns
    n = int(masks.max())
    tag = (proj_file.replace('_proj_', '').replace('.tif', '')
           + f'_d{int(diameter)}_ms{int(min_size)}_cp{cellprob_threshold:g}')
    out = f'_seg_labels_{tag}.npy'
    np.save(out, masks)
    # inline overlay
    bnd = find_boundaries(masks, mode='outer')
    ov = np.zeros((*masks.shape, 4)); ov[bnd] = (1, 0, 0, 1)
    plt.figure(figsize=(9, 9)); plt.imshow(img, cmap='gray'); plt.imshow(ov)
    plt.title(f'cpsam_v2 | {proj_file} | diam={eval_diam} min_size={min_size} cellprob={cellprob_threshold} | {n} cells')
    plt.axis('off'); plt.show()
    print(f'{proj_file}  diam={eval_diam} min_size={min_size} cellprob={cellprob_threshold}  ->  {n} cells  (saved {out})')
    return out

## 5) Run A — native scale (diameter = 0, no resize)
Cellpose-SAM is scale-invariant, so native is the fast, first-choice setting. Target ~200 cells.

In [ ]:
out_d0 = run('_proj_corr.tif', 0)

## 6) Run B — diameter = 8 (rescale small cells up to the trained size)
Our original approach. On GPU the ~3.75× upscale is still fast. Compare the cell count + separation against Run A.

In [ ]:
out_d8 = run('_proj_corr.tif', 8)

In [ ]:
# 6b) Diameter sweep 4-6 on the correlation projection.
# native (d0) covers the most cells but over-splits some; a mild upscale may fix that
# without the coverage collapse d8 caused. Reuses the model + uploaded file already in memory.
out_d4 = run('_proj_corr.tif', 4)
out_d6 = run('_proj_corr.tif', 6)
from google.colab import files
files.download(out_d4)
files.download(out_d6)

## 6c) Reduce over-detection at the source — `min_size` + `cellprob_threshold` sweep

The frozen hybrid's **splits (28)** and **junk (184)** come from CellPose over-detecting on the
native (d0) run (374 objects for ~200 cells). Two knobs cut that at the source:
- **`min_size`** ↑ (15 → 30/40): drops tiny fragments → fewer splits and fewer tiny junk.
- **`cellprob_threshold`** ↑ (0 → 1/2): stricter → fewer marginal junk detections.

Goal: bring the count down toward ~200–250 **without losing real cells**. Run a couple of settings,
eyeball the overlays, download the best one(s), and score locally (change `MASK_PATH` in the
detection notebook to the downloaded file).

In [ ]:
# 6c) Source-tuning sweep on the correlation projection (native scale).
# Reuses the model + uploaded _proj_corr.tif already in memory. Watch the cell count + overlay.
out_a = run('_proj_corr.tif', 0, min_size=30, cellprob_threshold=1.0)   # gentle
out_b = run('_proj_corr.tif', 0, min_size=40, cellprob_threshold=2.0)   # stricter

from google.colab import files
files.download(out_a)
files.download(out_b)

## 7) (Optional) same runs on the mean / max projections
Only if you uploaded them. Lets us pick the projection that separates the pairs best.

In [ ]:
# for pf in ['_proj_mean.tif', '_proj_max.tif']:
#     run(pf, 0); run(pf, 8)

## 8) Download the label masks
Bring these `.npy` files back to the repo, drop them in `data/results/comparisons/`, then locally
either (a) point the detection notebook's `MASK_PATH` at one and run it, or (b) score a mask
directly (segmentation only):
```bash
python cnmf_toolkit/cellpose_mask_to_A.py data/results/comparisons/_seg_labels_corr_d0.npy
python cnmf_toolkit/ground_truth_scorer.py \
  --annotation data/TaggedData/<your_tags>.tif \
  --run cellpose --stage fit/final --label cpsam-corr-d0
```
See `cnmf_toolkit/DL_SEEDING_WORKFLOW.md` for the full end-to-end workflow.

In [ ]:
from google.colab import files
files.download(out_d0)
files.download(out_d8)